# Netflix İzleme Geçmişi Panosu 1: Veriyi İndirip Temizlemek

[haticeozbolat.com](https://haticeozbolat.com) **Popüler Kültürü Kodluyorum** serisinin not defteri. Yazının tamamı: [YAZI LİNKİ]

Bu not defterinde kendi Netflix izleme geçmişini yükleyip temizliyor ve şu sorulara cevap arıyorsun:

- En çok izlediğin diziler hangileri?
- Hangi yıl, hangi gün, hangi ay daha çok izlemişsin?
- Maraton günün ve en uzun aran ne zaman?

**Gizlilik:** Yüklediğin dosya yalnızca bu Colab oturumunda durur, oturum kapanınca silinir. Dosyayı herkese açık bir yere yükleme.

## 1. İzleme geçmişini indir

1. Bir tarayıcıdan Netflix'te **Hesap** sayfasına git.
2. **Profiller**'den kendi profilini seç.
3. **İzleme etkinliği**'ni aç.
4. Sayfanın en altındaki **Tümünü indir** bağlantısına bas.

`NetflixViewingHistory.csv` adında bir dosya iner. Adımlar değişebilir; güncel hali için Netflix yardım sayfasına bak: https://help.netflix.com/en/node/101917

## 2. Dosyayı yükle

Aşağıdaki hücreyi çalıştırınca bir **Dosya seç** düğmesi çıkar. Elinde dosya yoksa hücreyi atla: not defteri hayali bir örnek veriyle devam eder.

In [ ]:
import os
import pandas as pd

DOSYA = "NetflixViewingHistory.csv"

try:
    from google.colab import files
    yuklenen = files.upload()  # Dosya seç penceresi açılır
    if yuklenen:
        DOSYA = next(iter(yuklenen))
except ImportError:
    pass  # Colab dışında çalışıyorsan dosyayı bu not defterinin yanına koy

if not os.path.exists(DOSYA):
    # Dosya yoksa hayali bir örnek veri üret (gerçek kişiye ait değildir)
    import numpy as np
    rng = np.random.default_rng(42)
    diziler = {"Uzay Kafesi": 12, "Kasabanın Sırrı": 8, "Mutfak Savaşları": 20}
    satirlar = []
    for dizi, bolum in diziler.items():
        for b in range(1, bolum + 1):
            satirlar.append(f"{dizi}: Sezon 1: Bölüm {b}")
    satirlar += ["Gece Treni", "Son Mektup", "Kayıp Harita"]
    tarihler = pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 600, len(satirlar)), unit="D")
    pd.DataFrame({"Title": satirlar, "Date": tarihler.strftime("%-m/%-d/%y")}).to_csv("ornek.csv", index=False)
    DOSYA = "ornek.csv"
    print("Dosya bulunamadı, hayali örnek veri kullanılıyor.")

print("Kullanılan dosya:", DOSYA)

## 3. Dosyayı oku

In [ ]:
df = pd.read_csv(DOSYA)
df = df.rename(columns={"Title": "baslik", "Date": "tarih"})
print(len(df), "kayıt")
df.head()

## 4. Tarihleri düzelt

Netflix tarihi hesabının diline göre farklı yazar: `21.10.2026` ya da `10/21/26` gibi. Bu fonksiyon ikisini de tanır. `errors="coerce"` okunamayan tarihi boş bırakır.

In [ ]:
def tarihi_coz(seri):
    ornek = seri.dropna().astype(str)
    if ornek.str.contains(r"\.").any():
        return pd.to_datetime(seri, format="%d.%m.%Y", errors="coerce")
    # Eğik çizgili biçimde ilk sayı 12'den büyükse gün önce gelir
    parca = ornek.str.split("/")
    gun_once = parca.str[0].astype(int).gt(12).any()
    yil_iki_hane = parca.str[-1].str.len().eq(2).all()
    if gun_once:
        bicim = "%d/%m/%y" if yil_iki_hane else "%d/%m/%Y"
    else:
        bicim = "%m/%d/%y" if yil_iki_hane else "%m/%d/%Y"
    return pd.to_datetime(seri, format=bicim, errors="coerce")

df["tarih"] = tarihi_coz(df["tarih"])
df = df.dropna(subset=["tarih"])
print(df["tarih"].min().date(), "→", df["tarih"].max().date())

## 5. Diziyi filmden ayır

Bir dizinin her bölümü ayrı satır olarak gelir: `Dizi Adı: Sezon 1: Bölüm 3`. İki adımda ayırıyoruz:

1. "Sezon", "Episode" gibi bir kelimeyle başlayan ilk parçaya kadar olan kısım dizi adıdır.
2. Hâlâ film sayılan bir başlığın iki noktadan önceki kısmı birden fazla kayıtta geçiyorsa o da dizidir.

In [ ]:
import re

SEZON = re.compile(
    r"^(Sezon|Season|Bölüm|Episode|Kısıtlı Dizi|Limited Series|Mini Dizi|Kitap|Book|"
    r"Part|Kısım|Volume|Class|Chapter|Collection)\b",
    re.IGNORECASE,
)

def ayir(baslik):
    parcalar = baslik.split(": ")
    for i, parca in enumerate(parcalar[1:], start=1):
        if SEZON.match(parca):
            return ": ".join(parcalar[:i]), "Dizi"
    return baslik, "Film"

df[["isim", "tur"]] = df["baslik"].apply(lambda b: pd.Series(ayir(b)))
print("İlk adımdan sonra film sayısı:", (df["tur"] == "Film").sum())

on_ek = df["baslik"].str.split(": ").str[0].str.strip()
tekrar = on_ek.map(on_ek.value_counts())
dizi_mi = (df["tur"] == "Film") & df["baslik"].str.contains(": ") & (tekrar >= 2)
df.loc[dizi_mi, "isim"] = on_ek[dizi_mi]
df.loc[dizi_mi, "tur"] = "Dizi"
print("İkinci adımdan sonra film sayısı:", (df["tur"] == "Film").sum())

print(df["tur"].value_counts())
print("Farklı dizi sayısı:", df.loc[df["tur"] == "Dizi", "isim"].nunique())

## 6. Veriye soru sor

Not: Netflix yarım bıraktığın bölümü de kaydeder. Bu yüzden saydığımız şey bölüm değil **kayıt**.

### En çok izlediğin diziler

In [ ]:
df.loc[df["tur"] == "Dizi", "isim"].value_counts().head(5)

### Yıllara, günlere ve aylara göre

In [ ]:
import matplotlib.pyplot as plt

MURDUM, SEFTALI, BOGURTLEN = "#2B1036", "#F6B17A", "#C2185B"
GUNLER = ["Pazartesi", "Salı", "Çarşamba", "Perşembe", "Cuma", "Cumartesi", "Pazar"]

yillik = df["tarih"].dt.year.value_counts().sort_index()
gunluk_dagilim = df["tarih"].dt.dayofweek.map(lambda i: GUNLER[i]).value_counts().reindex(GUNLER, fill_value=0)

aylik = df.groupby(df["tarih"].dt.to_period("M")).size()
tum_aylar = pd.period_range(aylik.index.min(), aylik.index.max(), freq="M")
aylik = aylik.reindex(tum_aylar, fill_value=0)  # Hiç izlenmeyen ayları 0 olarak ekle

fig, eksen = plt.subplots(3, 1, figsize=(9, 11))
renk = [BOGURTLEN if v == yillik.max() else SEFTALI for v in yillik]
eksen[0].bar(yillik.index.astype(str), yillik.values, color=renk)
eksen[0].set_title("Yıllara göre izleme kaydı")
renk = [BOGURTLEN if v == gunluk_dagilim.max() else SEFTALI for v in gunluk_dagilim]
eksen[1].bar(gunluk_dagilim.index, gunluk_dagilim.values, color=renk)
eksen[1].set_title("Haftanın günlerine göre")
eksen[2].plot(aylik.index.to_timestamp(), aylik.values, color=BOGURTLEN, linewidth=2)
eksen[2].set_title("Aylara göre")
for e in eksen:
    e.spines[["top", "right"]].set_visible(False)
    e.tick_params(colors=MURDUM)
plt.tight_layout()
plt.show()

print("En yoğun ay:", aylik.idxmax(), aylik.max(), "kayıt")
print("Hiç izlenmeyen ay sayısı:", (aylik == 0).sum())

### Maraton günün ve en uzun aran

In [ ]:
gunluk = df.groupby(df["tarih"].dt.date).size()
print("Maraton günü:", gunluk.idxmax(), "-", gunluk.max(), "kayıt")
o_gun = df[df["tarih"].dt.date == gunluk.idxmax()]
print("O gün en çok:", o_gun["isim"].value_counts().head(1).to_dict())

gunler = pd.Series(sorted(df["tarih"].dt.normalize().unique()))
aralar = gunler.diff()
en_uzun = aralar.idxmax()
print("En uzun ara:", aralar.max().days, "gün",
      gunler[en_uzun - 1].date(), "→", gunler[en_uzun].date())

### İzleme listeni yıl sonuna kadar bitirebilir misin?

`LISTEDEKI_YAPIM` sayısını kendi listene göre değiştir. Hesap kaba bir tahmin: son 30 günün hızıyla devam ettiğini varsayar.

In [ ]:
LISTEDEKI_YAPIM = 40  # Kendi izleme listendeki bölüm sayısını yaz
YIL_SONU = pd.Timestamp(f"{df['tarih'].max().year}-12-31")

son_30_gun = df[df["tarih"] >= df["tarih"].max() - pd.Timedelta(days=30)]
gunluk_hiz = len(son_30_gun) / 30
kalan_gun = (YIL_SONU - df["tarih"].max()).days
tahmin = gunluk_hiz * kalan_gun
print(f"Son 30 günde {len(son_30_gun)} kayıt, günde ortalama {gunluk_hiz:.1f}")
print(f"Bu hızla yıl sonuna kadar yaklaşık {tahmin:.0f} kayıt daha")
print("Yetişir gibi görünüyor!" if tahmin >= LISTEDEKI_YAPIM else "Biraz hızlanman gerekebilir.")

## Sırada ne var?

Netflix Panosu 2'de bu kodu Streamlit ile tıklanabilir bir panoya çevireceğiz.

Paylaşırken dikkat: ham CSV dosyasını herkese açık bir yere yükleme; yalnızca toplamları ve grafikleri paylaş.